# Hybrid Sim with Parental Tracking 

In [1]:
import random
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
from matplotlib.lines import Line2D
from typing import List, Tuple, Dict, Any, Optional, Literal
import matplotlib.pyplot as plt
import matplotlib.patches as patches # For drawing rectangles (locus blocks)
import numpy as np
from typing import List, Tuple, Dict, Any, Optional
import random

# --- Global Counters and Data Storage ---
# Global counter for unique individual IDs
individual_id_counter = 1

# Global lists to store detailed simulation data for analysis
all_locus_genotype_data = [] # Stores genotype and ancestry info for each locus of every individual
all_chromatid_recombination_data = [] # Stores recombination block data for each chromatid

# --- Constants for Allele Types ---
# Define numerical representations for alleles for consistency
ALLELE_YELLOW = 0  # Represents one pure population's allele (e.g., 'Y')
ALLELE_MAGENTA = 2 # Represents the other pure population's allele (e.g., 'M')
ALLELE_HETEROZYGOUS = 1 # Represents a heterozygous state at the diploid level (0 and 2)

In [2]:
class Chromosome:
    def __init__(self, alleles: List[int], ancestries: List[Tuple[str, int]]):
        """
        Represents a single chromosome strand as a list of integer alleles
        and a parallel list tracking the (pure_pop_label, founder_individual_id)
        origin of each locus.

        Args:
            alleles (List[int]): Alleles must now be integers: 0 (YELLOW), 2 (MAGENTA).
            ancestries (List[Tuple[str, int]]): For each allele, a tuple indicating
                                                 its original pure population label
                                                 and the ID of the founder individual.
                                                 E.g., ('P_A', 1), ('P_B', 5).
        """
        if len(alleles) != len(ancestries):
            raise ValueError("Alleles and ancestries lists must have the same length.")
        self.alleles = alleles
        self.ancestries = ancestries

    def __repr__(self) -> str:
        """
        Shows a preview of the first 10 numeric alleles and their ancestries.
        """
        allele_snippet = ''.join(map(str, self.alleles[:10])) if self.alleles else ''
        # For ancestry snippet, convert tuples to a more readable form, e.g., 'A1' from ('P_A', 1)
        ancestry_snippet_parts = []
        for anc in self.ancestries[:10]:
            if anc: # Check if tuple is not empty/None
                # Assuming anc is (pop_label, founder_id) e.g., ('P_A', 123)
                ancestry_snippet_parts.append(f"{anc[0][-1]}{anc[1]}") # Takes 'A' from 'P_A' and the ID
            else:
                ancestry_snippet_parts.append('?') # Placeholder for missing/uninitialized ancestry
        ancestry_snippet = ''.join(ancestry_snippet_parts)

        return f"Chr(Alleles:{allele_snippet}..., Ancestries:{ancestry_snippet}...)"


class DiploidChromosomePair:
    def __init__(self, chromatid1: Chromosome, chromatid2: Chromosome):
        """
        Represents a pair of homologous chromosomes (chromatids) in a diploid organism.

        Args:
            chromatid1 (Chromosome): The first chromatid.
            chromatid2 (Chromosome): The second chromatid, homologous to the first.
        """
        self.chromatid1 = chromatid1
        self.chromatid2 = chromatid2

    def __repr__(self) -> str:
        return f"DiploidPair(\n  {self.chromatid1},\n  {self.chromatid2}\n)"


def genotype_numeric(allele1: int, allele2: int) -> int:
    """
    Determines the numerical genotype from two alleles.
    - 0 if both alleles are 0 (homozygous 'Y')
    - 1 if one allele is 0 and the other is 2 (heterozygous 'YM')
    - 2 if both alleles are 2 (homozygous 'M')

    Args:
        allele1 (int): The integer value of the first allele (0 or 2).
        allele2 (int): The integer value of the second allele (0 or 2).

    Returns:
        int: The numerical representation of the genotype (0, 1, or 2).
    """
    if {allele1, allele2} == {ALLELE_YELLOW}:
        return 0
    elif {allele1, allele2} == {ALLELE_MAGENTA}:
        return 2
    elif {allele1, allele2} == {ALLELE_YELLOW, ALLELE_MAGENTA}:
        return 1
    else:
        raise ValueError(f"Invalid allele combination: {allele1}, {allele2}. Expected only {ALLELE_YELLOW} or {ALLELE_MAGENTA}.")


class Individual:
    def __init__(
        self,
        num_chromosomes: int,
        num_loci_per_chromosome: int,
        # Add these two new parameters
        parent1_id: Optional[int] = None,
        parent2_id: Optional[int] = None
    ):
        """
        Represents a single individual organism with a unique ID and a set of
        diploid chromosome pairs.

        Args:
            num_chromosomes (int): The number of diploid chromosome pairs this individual has.
            num_loci_per_chromosome (int): The number of loci on each chromosome.
            parent1_id (Optional[int]): The ID of the first parent.
            parent2_id (Optional[int]): The ID of the second parent.
        """
        global individual_id_counter
        self.id = individual_id_counter
        individual_id_counter += 1

        self.num_chromosomes = num_chromosomes
        self.num_loci_per_chromosome = num_loci_per_chromosome
        self.diploid_chromosome_pairs: List[DiploidChromosomePair] = []

        # Store the parent IDs as instance attributes
        self.parent1_id = parent1_id
        self.parent2_id = parent2_id

        # Optional attributes to store founder info if this individual is pure founder
        self._pure_pop_label: Optional[str] = None

    def get_all_numeric_genotypes(self) -> List[int]:
        """
        Retrieves the numerical genotype (0, 1, or 2) for each locus across all
        chromosome pairs of the individual.

        Returns:
            List[int]: A list of numerical genotypes for all loci.
        """
        all_numeric = []
        for pair in self.diploid_chromosome_pairs:
            alleles_chromatid1 = pair.chromatid1.alleles
            alleles_chromatid2 = pair.chromatid2.alleles
            for i in range(self.num_loci_per_chromosome):
                a1 = alleles_chromatid1[i]
                a2 = alleles_chromatid2[i]
                all_numeric.append(genotype_numeric(a1, a2))
        return all_numeric

    def calculate_hybrid_index(self) -> float:
        """
        Calculates the Hybrid Index (HI) for the individual, which is the proportion
        of alleles derived from the 'Magenta' parent (ALLELE_MAGENTA = 2) across all loci.

        Returns:
            float: The calculated Hybrid Index, ranging from 0.0 to 1.0.
        """
        genotypes = self.get_all_numeric_genotypes()
        if not genotypes:
            return 0.0
        # For diploid loci, the sum of numeric genotypes gives 2*number of 'M' alleles.
        # Total possible alleles are 2 * number of loci.
        total_possible_alleles = 2 * len(genotypes) # Each locus has 2 alleles
        total_magenta_alleles_count = sum(genotypes) # genotype_numeric(2,2)=2, genotype_numeric(0,2)=1, genotype_numeric(0,0)=0
        return total_magenta_alleles_count / total_possible_alleles

    def calculate_heterozygosity(self) -> float:
        """
        Calculates the overall heterozygosity for the individual, which is the proportion
        of heterozygous loci across all chromosome pairs.

        Returns:
            float: The proportion of heterozygous loci, ranging from 0.0 to 1.0.
        """
        genotypes = self.get_all_numeric_genotypes()
        if not genotypes:
            return 0.0
        # A genotype of 1 indicates heterozygosity (e.g., one ALLELE_YELLOW and one ALLELE_MAGENTA)
        return genotypes.count(ALLELE_HETEROZYGOUS) / len(genotypes)

    def get_chromatid_block_data(self):
        """
        Analyzes and returns recombination block data for each chromatid within the individual,
        now tracking both allele value blocks and ancestral origin blocks.

        Returns:
            List[Dict[str, Any]]: A list of dictionaries, each describing the blocks
                                  on a single chromatid.
        """
        all_chromatid_data = []
        chromatid_labels = ['_C1', '_C2'] # Suffix for chromatid within pair

        for chr_idx, diploid_pair in enumerate(self.diploid_chromosome_pairs):
            chromatids_in_pair = [diploid_pair.chromatid1, diploid_pair.chromatid2]

            for i, chromatid in enumerate(chromatids_in_pair):
                alleles = chromatid.alleles
                ancestries = chromatid.ancestries # Get ancestries list too!

                # Analyze allele blocks
                junctions_alleles, lengths_alleles, allele_vals = self._analyse_single_chromatid_sequence(alleles)

                # Analyze ancestry blocks
                junctions_ancestries, lengths_ancestries, ancestry_vals = self._analyse_single_chromatid_sequence(ancestries)

                # Prepare ancestry block values for easier understanding, e.g., 'A_ID123'
                formatted_ancestry_vals = []
                for anc_tuple in ancestry_vals:
                    if anc_tuple:
                        formatted_ancestry_vals.append(f"{anc_tuple[0][-1]}_ID{anc_tuple[1]}")
                    else:
                        formatted_ancestry_vals.append("UNKNOWN") # Should not happen if initialized correctly

                all_chromatid_data.append({
                    'individual_id': self.id,
                    'diploid_chr_id': chr_idx + 1,
                    'chromatid_in_pair_label': f"chr{chr_idx+1}{chromatid_labels[i]}", # e.g. 'chr1_C1'
                    'total_junctions_alleles': junctions_alleles,
                    'block_lengths_alleles': lengths_alleles,
                    'block_alleles': allele_vals,
                    'total_junctions_ancestries': junctions_ancestries,
                    'block_lengths_ancestries': lengths_ancestries,
                    'block_ancestries_formatted': formatted_ancestry_vals # Store formatted ancestry
                })
        return all_chromatid_data

    def _analyse_single_chromatid_sequence(self, sequence: List[Any]) -> Tuple[int, List[int], List[Any]]:
        """
        Helper method to identify consecutive blocks of identical values in a sequence
        (either alleles or ancestries).

        Args:
            sequence (List[Any]): The list of alleles or ancestries to analyze.

        Returns:
            Tuple[int, List[int], List[Any]]: A tuple containing:
                - The number of junctions (block changes).
                - A list of block lengths.
                - A list of the values/types for each block.
        """
        if not sequence:
            return 0, [], []

        block_lengths = []
        block_values = []

        # Use itertools.groupby to find consecutive identical values
        for value, group in itertools.groupby(sequence):
            block = list(group)
            block_lengths.append(len(block))
            block_values.append(value)

        junctions = len(block_lengths) - 1
        return junctions, block_lengths, block_values

In [3]:
def meiosis_with_recombination(
    diploid_pair: DiploidChromosomePair,
    recomb_event_probabilities: Dict[int, float],
    recomb_probabilities: List[float]
) -> Chromosome:
    """
    Simulates meiosis with a variable number of recombination events for one chromosome pair,
    now also tracking ancestral origin for each locus.

    Args:
        diploid_pair (DiploidChromosomePair): The pair of homologous chromatids.
        recomb_event_probabilities (dict): Probability for 0, 1, or 2 recombination events
                                           (keys are number of crossovers, values are probabilities).
        recomb_probabilities (list): Position-dependent probabilities for recombination occurring
                                     at each specific locus position along a chromosome (length N-1 for N loci).

    Returns:
        Chromosome: A recombinant chromosome with alleles and ancestral origins after meiosis.
    """
    loci_len = len(diploid_pair.chromatid1.alleles)

    # Determine the number of recombination events based on provided probabilities
    n_events = random.choices(
        population=[0, 1, 2], # Assuming max 2 crossovers for simplicity as per common models
        weights=[recomb_event_probabilities.get(i, 0) for i in [0, 1, 2]],
        k=1
    )[0]

    chosen_positions = []
    if n_events > 0:
        # Generate possible recombination points (between loci)
        # Positions are 1-based, indicating the point *after* which recombination occurs
        # e.g., position 1 means between locus 0 and locus 1
        possible_recomb_points = list(range(1, loci_len)) # There are (loci_len - 1) possible points

        # If there are no recombination probabilities or all are zero, fall back to random uniform selection
        if not recomb_probabilities or sum(recomb_probabilities) == 0:
            if len(possible_recomb_points) < n_events:
                chosen_positions = possible_recomb_points[:] # Take all if not enough
            else:
                chosen_positions = sorted(random.sample(possible_recomb_points, n_events))
        else:
            # Select recombination points based on recombination_probabilities
            # Ensure recomb_probabilities has enough entries (loci_len - 1)
            if len(recomb_probabilities) < loci_len -1:
                 raise ValueError("recomb_probabilities length must be at least num_loci_per_chr - 1")

            # Adjust weights to match the actual number of possible recombination points
            weights = recomb_probabilities[:loci_len-1] # Use only relevant part of the list
            if sum(weights) == 0: # Handle case where provided weights are all zero but n_events > 0
                if len(possible_recomb_points) < n_events:
                    chosen_positions = possible_recomb_points[:]
                else:
                    chosen_positions = sorted(random.sample(possible_recomb_points, n_events))
            else:
                num_events_to_pick = min(n_events, len(possible_recomb_points))
                while len(chosen_positions) < num_events_to_pick:
                    # random.choices picks from population with replacement by default.
                    # We need to ensure unique positions if not picking all.
                    # For a small number of events, picking one by one and checking for uniqueness is fine.
                    chosen_point = random.choices(possible_recomb_points, weights=weights, k=1)[0]
                    if chosen_point not in chosen_positions:
                        chosen_positions.append(chosen_point)
                chosen_positions.sort()

    # Determine starting chromatid (randomly choose which parent's chromatid contributes the first segment)
    if random.random() < 0.5:
        current_alleles_source = diploid_pair.chromatid1.alleles
        current_ancestries_source = diploid_pair.chromatid1.ancestries
        other_alleles_source = diploid_pair.chromatid2.alleles
        other_ancestries_source = diploid_pair.chromatid2.ancestries
    else:
        current_alleles_source = diploid_pair.chromatid2.alleles
        current_ancestries_source = diploid_pair.chromatid2.ancestries
        other_alleles_source = diploid_pair.chromatid1.alleles
        other_ancestries_source = diploid_pair.chromatid1.ancestries

    recombinant_alleles = []
    recombinant_ancestries = []
    last_pos = 0
    # Add loci_len as a breakpoint to ensure the last segment is included
    breakpoints = sorted(chosen_positions + [loci_len])

    for pos in breakpoints:
        # Extend the recombinant chromatid with the segment from the current source
        recombinant_alleles.extend(current_alleles_source[last_pos:pos])
        recombinant_ancestries.extend(current_ancestries_source[last_pos:pos])

        # Switch source chromatids for the next segment (if not at the end)
        if pos != loci_len:
            current_alleles_source, other_alleles_source = other_alleles_source, current_alleles_source
            current_ancestries_source, other_ancestries_source = other_ancestries_source, current_ancestries_source
        last_pos = pos

    return Chromosome(recombinant_alleles, recombinant_ancestries)

In [4]:
def record_individual_genome(individual: 'Individual', generation_label: str):
    """
    Records the full genotype and ancestral origin of each locus for every chromosome pair
    within a given individual. This data is then appended to the global `all_locus_genotype_data` list.

    Each entry in `all_locus_genotype_data` is a dictionary providing:
      - 'generation': The specific generation label (e.g., 'F2', 'BC1A').
      - 'individual_id': The unique identifier for the individual.
      - 'diploid_chr_id': The chromosome pair number (1-based for clarity).
      - 'locus_position': The position index of the locus along the chromosome (0-based).
      - 'genotype_allele_A': Allele value on chromatid 1 (int).
      - 'genotype_allele_B': Allele value on chromatid 2 (int).
      - 'ancestry_chromatid1_pop': Pure population label for chromatid 1's allele (str, e.g., 'P_A').
      - 'ancestry_chromatid1_founder_id': Founder ID for chromatid 1's allele (int).
      - 'ancestry_chromatid2_pop': Pure population label for chromatid 2's allele (str, e.g., 'P_B').
      - 'ancestry_chromatid2_founder_id': Founder ID for chromatid 2's allele (int).
      - 'parent1_id': The unique ID of the first parent.
      - 'parent2_id': The unique ID of the second parent.

    Args:
        individual (Individual): The 'Individual' object whose genome I want to record.
        generation_label (str): A string label to associate with the current generation.
    """
    for chr_idx, pair in enumerate(individual.diploid_chromosome_pairs):
        alleles_chromatid1 = pair.chromatid1.alleles
        alleles_chromatid2 = pair.chromatid2.alleles
        ancestries_chromatid1 = pair.chromatid1.ancestries
        ancestries_chromatid2 = pair.chromatid2.ancestries

        for locus_idx in range(individual.num_loci_per_chromosome):
            allele_a = alleles_chromatid1[locus_idx]
            allele_b = alleles_chromatid2[locus_idx]

            # Ancestry will be a tuple (pure_pop_label, founder_id)
            anc_a_tuple = ancestries_chromatid1[locus_idx]
            anc_b_tuple = ancestries_chromatid2[locus_idx]

            # Extract components for separate columns
            anc_a_pop = anc_a_tuple[0] if anc_a_tuple else None
            anc_a_founder_id = anc_a_tuple[1] if anc_a_tuple else None
            anc_b_pop = anc_b_tuple[0] if anc_b_tuple else None
            anc_b_founder_id = anc_b_tuple[1] if anc_b_tuple else None

            all_locus_genotype_data.append({
                'generation': generation_label,
                'individual_id': individual.id,
                'diploid_chr_id': chr_idx + 1,
                'locus_position': locus_idx,
                'genotype_allele_A': allele_a,
                'genotype_allele_B': allele_b,
                'ancestry_chromatid1_pop': anc_a_pop,
                'ancestry_chromatid1_founder_id': anc_a_founder_id,
                'ancestry_chromatid2_pop': anc_b_pop,
                'ancestry_chromatid2_founder_id': anc_b_founder_id,
                'parent1_id': individual.parent1_id, # ADDED: New parent column
                'parent2_id': individual.parent2_id  # ADDED: New parent column
            })


def record_chromatid_recombination(individual: 'Individual', generation_label: str):
    """
    This function records the detailed recombination block data for an individual's chromatids,
    now including both allele and ancestry block information.
    The data is appended to the global `all_chromatid_recombination_data` list.

    Args:
        individual (Individual): The 'Individual' object whose chromatid recombination
                                 data I want to record.
        generation_label (str): A string label to associate with the current generation.
    """
    chromatid_data = individual.get_chromatid_block_data()
    for record in chromatid_data:
        record['generation'] = generation_label
        all_chromatid_recombination_data.append(record)

In [5]:
def create_pure_individual_with_ancestry(num_chromosomes: int, num_loci_per_chr: int, allele_type: Any, pure_pop_label: str, individual_obj: 'Individual') -> 'Individual':
    """
    Creates a single pure homozygous individual, initialising both alleles and their specific ancestral origin.
    This version takes an already created Individual object to use its assigned ID.

    Args:
        num_chromosomes (int): Number of chromosome pairs.
        num_loci_per_chr (int): Number of loci per chromosome.
        allele_type (Any): Allele value (e.g., ALLELE_YELLOW, ALLELE_MAGENTA, or their string representations).
        pure_pop_label (str): The label of the pure population ('P_A' or 'P_B').
        individual_obj (Individual): An already initialized Individual object, so its ID can be used for ancestry tagging.

    Returns:
        Individual: The passed Individual object, now populated with homozygous alleles and ancestral origins.
    """
    # Convert allele_type to integer if it's a string
    if isinstance(allele_type, str):
        if allele_type == '0' or allele_type.upper() == 'Y':
            initial_allele_int = ALLELE_YELLOW
        elif allele_type == '1': # This seems like an intermediate state if 0 and 2 are pure
            initial_allele_int = ALLELE_HETEROZYGOUS # Consider if this case is truly "pure"
        elif allele_type == '2' or allele_type.upper() == 'M':
            initial_allele_int = ALLELE_MAGENTA
        else:
            raise ValueError(f"Unsupported allele_type string: {allele_type}. Expected '0', '1', '2', 'Y', or 'M'.")
    elif isinstance(allele_type, int):
        if allele_type in (ALLELE_YELLOW, ALLELE_HETEROZYGOUS, ALLELE_MAGENTA):
            initial_allele_int = allele_type
        else:
            raise ValueError(f"Unsupported allele_type integer: {allele_type}. Expected 0, 1, or 2.")
    else:
        raise TypeError(f"allele_type must be str or int, got {type(allele_type)}")

    individual = individual_obj # Use the pre-created individual

    # Store the pure population label on the individual for easy access if needed
    individual._pure_pop_label = pure_pop_label

    # Create the ancestry tuple for all loci of this founder
    # This captures the specific pure population and the unique ID of this founder individual
    founder_ancestry_tag = (pure_pop_label, individual.id)

    # Clear any pre-existing chromosome pairs if the individual object was reused or partially initialized
    individual.diploid_chromosome_pairs = []

    for _ in range(num_chromosomes):
        chromosome_alleles = [initial_allele_int] * num_loci_per_chr
        # Each allele's ancestry is set to the current founder's tag
        chromosome_ancestries = [founder_ancestry_tag] * num_loci_per_chr

        chromatid1 = Chromosome(chromosome_alleles[:], chromosome_ancestries[:])
        chromatid2 = Chromosome(chromosome_alleles[:], chromosome_ancestries[:])

        individual.diploid_chromosome_pairs.append(DiploidChromosomePair(chromatid1, chromatid2))

    return individual


def create_pure_populations(
    num_individuals: int,
    num_chromosomes: int,
    num_loci_per_chr: int,
    allele_type: Any,
    pure_pop_label: str
) -> List['Individual']:
    """
    Creates a population of pure homozygous individuals, each with a unique founder ID.
    The ancestry of all loci on their chromosomes is tagged with this ID and the population label.

    Args:
        num_individuals (int): The number of individuals to create in this pure population.
        num_chromosomes (int): The number of diploid chromosome pairs per individual.
        num_loci_per_chr (int): The number of loci per chromosome.
        allele_type (Any): The allele type (0, 2, 'Y', 'M') that defines this pure population.
        pure_pop_label (str): The label for this pure population (e.g., 'P_A', 'P_B').

    Returns:
        List[Individual]: A list of newly created pure individuals.
    """
    pure_population = []
    for _ in range(num_individuals):
        # 1. Create an Individual object first. This automatically assigns it a unique global ID.
        new_individual = Individual(num_chromosomes=num_chromosomes, num_loci_per_chromosome=num_loci_per_chr)

        # 2. Populate this individual's chromosomes with the correct alleles and ancestry tags
        #    using its newly assigned unique ID.
        populated_individual = create_pure_individual_with_ancestry(
            num_chromosomes,
            num_loci_per_chr,
            allele_type,
            pure_pop_label,
            new_individual
        )
        pure_population.append(populated_individual)
    return pure_population

In [6]:
def build_forward_generations(base_name: str, start_gen: int, end_gen: int) -> List[Tuple[str, str, str]]:
    """
    I use this function to create a breeding plan for sequential forward generations (e.g., F1, F2, F3...).
    The process starts from 'start_gen' and goes up to 'end_gen' (inclusive).
    The very first generation (specified by 'start_gen') is always a cross between two pure parental populations ('P_A' and 'P_B').
    Subsequent generations in this forward sequence are then bred by crossing individuals from the *previous* generation amongst themselves.

    Args:
        base_name (str): The prefix for the generation names (e.g., "F" for Filial generations).
        start_gen (int): The starting generation number (e.g., 1 for F1).
        end_gen (int): The final generation number to include (e.g., 5 for F5).

    Returns:
        List[Tuple[str, str, str]]: A list of tuples, where each tuple represents a planned cross:
                                    (new_generation_label, parent1_label, parent2_label).
    """
    plan = [] # Initialise an empty list to store my breeding plan.
    for i in range(start_gen, end_gen + 1):
        current_gen_label = f"{base_name}{i}" # Construct the label for the current generation, e.g., "F1", "F2".
        if i == start_gen:
            # For the first generation in the sequence, I'm crossing the pure parental populations.
            plan.append((current_gen_label, 'P_A', 'P_B'))
        else:
            # For subsequent generations, I cross individuals from the previous generation with themselves.
            previous_gen_label = f"{base_name}{i-1}"
            plan.append((current_gen_label, previous_gen_label, previous_gen_label))
    return plan # Return the complete breeding plan.


def build_backcross_generations(
    base_name: str,
    initial_hybrid_gen_label: str, # This will be 'F1' (or whatever starts the BC series)
    pure_pop_label: str,
    num_backcross_generations: int # How many BC generations do you want (e.g., 5 for BC1, BC2, BC3, BC4, BC5)
) -> List[Tuple[str, str, str]]:
    """
    This function builds a sequential backcross generation plan.
    BC1 = initial_hybrid_gen_label x pure_pop_label
    BC2 = BC1 x pure_pop_label
    ...
    BCn = BC(n-1) x pure_pop_label

    Args:
        base_name (str): The prefix for backcross generation names (e.g., "BC").
        initial_hybrid_gen_label (str): The label of the first hybrid generation to be backcrossed
                                            (e.g., "F1").
        pure_pop_label (str): The label of the pure parental population (e.g., "P_A" or "P_B")
                                that the hybrid generations will be repeatedly crossed with.
        num_backcross_generations (int): The total number of backcross generations to create (e.g., 5 for BC1 to BC5).

    Returns:
        List[Tuple[str, str, str]]: A list of backcross generation crosses.
                                    Example: [('BC1A', 'F1', 'P_A'), ('BC2A', 'BC1A', 'P_A'), ...]
    """
    plan = []
    # The recurrent parent is always the pure population
    recurrent_parent = pure_pop_label

    # The first parent for BC1 is the initial hybrid generation (e.g., F1)
    current_hybrid_parent = initial_hybrid_gen_label

    # Iterate to create the desired number of backcross generations
    for i in range(1, num_backcross_generations + 1):
        # Construct the label for the current backcross generation, e.g., "BC1A", "BC2A"
        # The pure_pop_label[-1] correctly extracts 'A' or 'B' from 'P_A' or 'P_B'
        backcross_label = f"{base_name}{i}{pure_pop_label[-1]}"

        # Append the planned cross: (new BC generation, hybrid parent, recurrent parent)
        plan.append((backcross_label, current_hybrid_parent, recurrent_parent))

        # For the next iteration, the newly created backcross generation becomes the hybrid parent
        current_hybrid_parent = backcross_label

    return plan

In [7]:
def run_genetic_cross(
    parents_pop_A: List['Individual'],
    parents_pop_B: List['Individual'],
    offspring_count_per_mating_pair: int,
    generation_label: str,
    num_chromosomes_for_offspring: int,
    recomb_event_probabilities: Dict[int, float],
    recomb_probabilities: List[float]
) -> List['Individual']:
    """
    I use this function to simulate a genetic cross, where individuals from two distinct parental
    populations (pop_A and pop_B) mate to produce offspring.
    Each unique mating pair will produce 'offspring_count_per_mating_pair' offspring.

    Args:
        parents_pop_A (List[Individual]): The first group of parental individuals available for mating.
        parents_pop_B (List[Individual]): The second group of parental individuals available for mating.
        offspring_count_per_mating_pair (int): The number of new offspring individuals I want to generate
                                                *for each unique mating pair*.
        generation_label (str): A descriptive label for the new generation being created (e.g., "F2", "BC1A").
        num_chromosomes_for_offspring (int): The number of diploid chromosome pairs each new offspring will have.
        recomb_event_probabilities (dict): A probability distribution that dictates how many
                                            recombination events (crossovers) occur on a chromosome during meiosis.
        recomb_probabilities (List[float]): A list or array of probabilities for recombination occurring
                                            at each specific locus position along a chromosome.

    Returns:
        List[Individual]: A list containing all the newly created offspring individuals from this cross.
    """
    # Debug prints (keep these for now, they are very helpful!)
    print(f"\n--- DEBUG_CROSS for {generation_label} ---")
    print(f"DEBUG_CROSS: Parent A size entering cross: {len(parents_pop_A)}")
    print(f"DEBUG_CROSS: Parent B size entering cross: {len(parents_pop_B)}")
    print(f"DEBUG_CROSS: Offspring *per mating pair* expected: {offspring_count_per_mating_pair}")

    offspring = [] # Initialise an empty list to store the new individuals.

    # --- UPDATED LOGIC TO PREVENT SELF-CROSSING ---
    mating_pairs = []

    # Check if the parent populations are the same (e.g., F1 x F1 cross)
    if parents_pop_A is parents_pop_B:
        # For within-population crosses, shuffle the list and pair adjacent individuals
        shuffled_parents = random.sample(parents_pop_A, len(parents_pop_A))
        # Take pairs from the shuffled list, handling odd numbers of individuals
        for i in range(0, len(shuffled_parents) // 2 * 2, 2):
            mating_pairs.append((shuffled_parents[i], shuffled_parents[i+1]))
    else:
        # For between-population crosses (e.g., P_A x P_B), use original zip logic
        shuffled_parent_A = random.sample(parents_pop_A, len(parents_pop_A))
        shuffled_parent_B = random.sample(parents_pop_B, len(parents_pop_B))
        num_mating_pairs = min(len(shuffled_parent_A), len(shuffled_parent_B))
        mating_pairs = list(zip(shuffled_parent_A[:num_mating_pairs], shuffled_parent_B[:num_mating_pairs]))

    print(f"DEBUG_CROSS: Number of unique mating pairs formed: {len(mating_pairs)}")

    # Iterate through all mating pairs
    for parent_A, parent_B in mating_pairs:
        # For EACH unique mating pair, create the specified number of offspring
        for _ in range(offspring_count_per_mating_pair):
            child = Individual(
                num_chromosomes=num_chromosomes_for_offspring,
                num_loci_per_chromosome=parent_A.num_loci_per_chromosome,
                parent1_id=parent_A.id,
                parent2_id=parent_B.id
            )

            for chr_idx in range(num_chromosomes_for_offspring):
                diploid_pair_A = parent_A.diploid_chromosome_pairs[chr_idx]
                diploid_pair_B = parent_B.diploid_chromosome_pairs[chr_idx]

                haploid_from_A = meiosis_with_recombination(diploid_pair_A, recomb_event_probabilities, recomb_probabilities)
                haploid_from_B = meiosis_with_recombination(diploid_pair_B, recomb_event_probabilities, recomb_probabilities)

                child.diploid_chromosome_pairs.append(DiploidChromosomePair(haploid_from_A, haploid_from_B))

            record_individual_genome(child, generation_label)
            record_chromatid_recombination(child, generation_label)

            offspring.append(child)

    print(f"DEBUG_CROSS: Final new_generation size created: {len(offspring)}")
    print(f"--- END DEBUG_CROSS for {generation_label} ---\n")

    return offspring

In [8]:
def calculate_hi_het_for_population(population: List['Individual']) -> List[Dict[str, float]]:
    """
    Calculates Hybrid Index (HI) and Heterozygosity (HET) for each individual in a given population.

    Args:
        population (List[Individual]): A list of Individual objects.

    Returns:
        List[Dict[str, float]]: A list of dictionaries, where each dictionary contains
                                 'id', 'HI', and 'HET' for an individual.
    """
    data = []
    for indiv in population:
        hi = indiv.calculate_hybrid_index()
        het = indiv.calculate_heterozygosity()
        # Make sure 'id' attribute exists on your Individual objects (it does after Cell 2 changes)
        data.append({'id': indiv.id, 'HI': hi, 'HET': het})
    return data

def population_stats(population: List['Individual']) -> Dict[str, float]:
    """
    Calculates summary statistics (mean, std dev) for Hybrid Index and Heterozygosity
    across a population.

    Args:
        population (List[Individual]): A list of Individual objects.

    Returns:
        Dict[str, float]: A dictionary containing mean and standard deviation for HI and HET.
    """
    if not population:
        return {'mean_HI': 0.0, 'std_HI': 0.0, 'mean_HET': 0.0, 'std_HET': 0.0}

    hi_values = [ind.calculate_hybrid_index() for ind in population]
    het_values = [ind.calculate_heterozygosity() for ind in population]

    mean_hi = np.mean(hi_values) if hi_values else 0.0
    std_hi = np.std(hi_values) if hi_values else 0.0
    mean_het = np.mean(het_values) if het_values else 0.0
    std_het = np.std(het_values) if het_values else 0.0

    return {
        'mean_HI': mean_hi,
        'std_HI': std_hi,
        'mean_HET': mean_het,
        'std_HET': std_het
    }


def simulate_generations(
    initial_pop_A: List['Individual'] = None,
    initial_pop_B: List['Individual'] = None,
    generation_plan: List[Tuple[str, str, str]] = None,
    num_offspring_per_cross: int = 2,
    num_chromosomes: int = 2,
    recomb_event_probabilities: Dict[int, float] = None,
    recomb_probabilities: List[float] = None,
    verbose: bool = False,
):
    """
    Simulates genetic crosses across multiple generations based on a predefined breeding plan.
    It tracks populations, Hybrid Index, Heterozygosity, and detailed genomic and recombination data.

    Args:
        initial_pop_A (List[Individual], optional): The initial pure population A. Defaults to None.
        initial_pop_B (List[Individual], optional): The initial pure population B. Defaults to None.
        generation_plan (List[Tuple[str, str, str]], optional): A list of tuples defining
                                                                 (new_gen_label, parent1_label, parent2_label).
                                                                 Defaults to None.
        num_offspring_per_cross (int): Number of offspring to generate per unique mating pair.
        num_chromosomes (int): Number of chromosome pairs per individual.
        recomb_event_probabilities (dict): Probability distribution for recombination events.
        recomb_probabilities (List[float]): Locus-specific recombination probabilities.
        verbose (bool): If True, prints detailed statistics for each generation. Defaults to False.

    Returns:
        Tuple[Dict[str, List[Individual]], Dict[str, List[Dict[str, float]]], List[Dict[str, Any]], List[Dict[str, Any]]]:
            - populations: Dictionary of generated populations {gen_label: List[Individual]}.
            - all_generations_data: Dictionary of HI/HET data per individual per generation.
            - all_locus_genotype_data: Global list of detailed locus-level genotype and ancestry data.
            - all_chromatid_recombination_data: Global list of detailed chromatid recombination data.
    """
    # Initialise populations dict
    populations = {} # Start fresh for each simulation run

    # Clear global data lists for a new simulation run
    global all_locus_genotype_data
    global all_chromatid_recombination_data
    all_locus_genotype_data = []
    all_chromatid_recombination_data = []

    # Initialise dict to store HI and HET data for each generation
    all_generations_data = {}

    # Add initial pure populations if provided, and record HI/HET for them
    if initial_pop_A is not None:
        populations['P_A'] = initial_pop_A
        for ind in initial_pop_A:
            record_individual_genome(ind, 'P_A')
            record_chromatid_recombination(ind, 'P_A')
        all_generations_data['P_A'] = calculate_hi_het_for_population(initial_pop_A)
        if verbose:
            stats = population_stats(initial_pop_A)
            print(f"P_A initialized | Count: {len(initial_pop_A)} | Mean HI: {stats['mean_HI']:.3f} (±{stats['std_HI']:.3f}), "
                  f"Mean HET: {stats['mean_HET']:.3f} (±{stats['std_HET']:.3f})")

    if initial_pop_B is not None:
        populations['P_B'] = initial_pop_B
        for ind in initial_pop_B:
            record_individual_genome(ind, 'P_B')
            record_chromatid_recombination(ind, 'P_B')
        all_generations_data['P_B'] = calculate_hi_het_for_population(initial_pop_B)
        if verbose:
            stats = population_stats(initial_pop_B)
            print(f"P_B initialized | Count: {len(initial_pop_B)} | Mean HI: {stats['mean_HI']:.3f} (±{stats['std_HI']:.3f}), "
                  f"Mean HET: {stats['mean_HET']:.3f} (±{stats['std_HET']:.3f})")

    # Check for generation plan
    if generation_plan is None:
        print("Warning: No generation plan provided. Returning initial populations.")
        return populations, all_generations_data, all_locus_genotype_data, all_chromatid_recombination_data

    # Loop over planned generations to simulate crosses
    for gen_info in generation_plan:
        if len(gen_info) < 3: # Ensure sufficient info (new_gen_label, parent1, parent2)
            print(f"Skipping malformed generation plan entry: {gen_info}. Expected (label, parent1, parent2).")
            continue

        gen_name = gen_info[0]
        parents_names = gen_info[1:] # Should be [parent1_label, parent2_label]

        # Check if parent populations exist
        try:
            parents_pop_A_for_cross = populations[parents_names[0]]
            parents_pop_B_for_cross = populations[parents_names[1]]
        except KeyError as e:
            print(f"Error: Parent population '{e}' not found for generation '{gen_name}'. Skipping this generation.")
            continue # Skip to the next generation in the plan

        # Run the cross to get new generation
        new_pop = run_genetic_cross(
            parents_pop_A_for_cross,
            parents_pop_B_for_cross,
            offspring_count_per_mating_pair=num_offspring_per_cross,
            generation_label=gen_name,
            num_chromosomes_for_offspring=num_chromosomes,
            recomb_event_probabilities=recomb_event_probabilities,
            recomb_probabilities=recomb_probabilities
        )

        # Store new population
        populations[gen_name] = new_pop

        # Calculate and store HI/HET for this generation
        all_generations_data[gen_name] = calculate_hi_het_for_population(new_pop)

        if verbose:
            stats = population_stats(new_pop)
            print(f"{gen_name} created from parents {parents_names[0]} and {parents_names[1]} | "
                  f"Count: {len(new_pop)} | Mean HI: {stats['mean_HI']:.3f} (±{stats['std_HI']:.3f}), "
                  f"Mean HET: {stats['mean_HET']:.3f} (±{stats['std_HET']:.3f})")
            print(f"Added '{gen_name}' to populations. Current population keys: {list(populations.keys())}")
            print("-" * 50) # Separator for verbose output

    return populations, all_generations_data, all_locus_genotype_data, all_chromatid_recombination_data

In [9]:
# Reset global individual_id_counter for a fresh simulation run to ensure unique IDs across runs
global individual_id_counter
individual_id_counter = 1

# 1. Define Simulation Parameters
num_individuals_per_pure_pop = 20 # Recommended for decent population sizes
num_offspring_per_cross = 1        # Recommended for maintaining population sizes

num_chromosomes = 2
num_loci_per_chr = 100

# Recombination parameters (from previous discussions)
# Example: 1 crossover per chromosome on average
recomb_event_probabilities = {0: 0.0, 1: 1.0, 2: 0.0} # Example distribution: exactly 1 crossover per chromosome
recomb_probabilities = [0.01] * (num_loci_per_chr - 1) # Low, uniform recombination probability between loci

# 2. Create Initial Pure Populations (P_A and P_B)
print("Creating initial pure populations (P_A and P_B)...")

# P_A individuals should have all alleles = ALLELE_MAGENTA (2)
pop_A = create_pure_populations(
    num_individuals_per_pure_pop,
    num_chromosomes,
    num_loci_per_chr,
    allele_type=ALLELE_MAGENTA, # Use integer 2 for P_A alleles
    pure_pop_label='P_A' # Pass the population label
)
print(f"P_A created with {len(pop_A)} individuals.")

# P_B individuals should have all alleles = ALLELE_YELLOW (0)
pop_B = create_pure_populations(
    num_individuals_per_pure_pop,
    num_chromosomes,
    num_loci_per_chr,
    allele_type=ALLELE_YELLOW, # Use integer 0 for P_B alleles
    pure_pop_label='P_B' # Pass the population label
)
print(f"P_B created with {len(pop_B)} individuals.")

# 3. Define Breeding Plans
print("\nDefining breeding plans for forward and backcross generations...")

# Forward generations
forward_plan = build_forward_generations(
    base_name='F',
    start_gen=1,
    end_gen=6
)

# Backcross generations (e.g., BC1A to BC20A, and BC1B to BC20B)
num_sequential_backcrosses = 0

backcross_plan_A = build_backcross_generations(
    base_name='BC',
    initial_hybrid_gen_label='F1',
    pure_pop_label='P_A',
    num_backcross_generations=num_sequential_backcrosses
)

backcross_plan_B = build_backcross_generations(
    base_name='BC',
    initial_hybrid_gen_label='F1',
    pure_pop_label='P_B',
    num_backcross_generations=num_sequential_backcrosses
)

# Combine all plans into a single comprehensive breeding plan
full_breeding_plan = forward_plan + backcross_plan_A + backcross_plan_B
print(f"Total generations in breeding plan: {len(full_breeding_plan)}")

# 4. Simulate Generations
print("\nStarting genetic simulation across generations...")
populations, all_generations_data, locus_data_list_global, recombination_data_list_global = simulate_generations(
    initial_pop_A=pop_A, # Pass P_A population
    initial_pop_B=pop_B, # Pass P_B population
    generation_plan=full_breeding_plan,
    num_offspring_per_cross=num_offspring_per_cross,
    num_chromosomes=num_chromosomes,
    recomb_event_probabilities=recomb_event_probabilities,
    recomb_probabilities=recomb_probabilities,
    verbose=True, # Set to True to see detailed stats per generation
)

print("\nSimulation complete!")
print(f"Final number of populations tracked: {len(populations)}")
# You can inspect specific population sizes, e.g., print(len(populations['F1']))

Creating initial pure populations (P_A and P_B)...
P_A created with 20 individuals.
P_B created with 20 individuals.

Defining breeding plans for forward and backcross generations...
Total generations in breeding plan: 6

Starting genetic simulation across generations...
P_A initialized | Count: 20 | Mean HI: 1.000 (±0.000), Mean HET: 0.000 (±0.000)
P_B initialized | Count: 20 | Mean HI: 0.000 (±0.000), Mean HET: 0.000 (±0.000)

--- DEBUG_CROSS for F1 ---
DEBUG_CROSS: Parent A size entering cross: 20
DEBUG_CROSS: Parent B size entering cross: 20
DEBUG_CROSS: Offspring *per mating pair* expected: 1
DEBUG_CROSS: Number of unique mating pairs formed: 20
DEBUG_CROSS: Final new_generation size created: 20
--- END DEBUG_CROSS for F1 ---

F1 created from parents P_A and P_B | Count: 20 | Mean HI: 0.500 (±0.000), Mean HET: 1.000 (±0.000)
Added 'F1' to populations. Current population keys: ['P_A', 'P_B', 'F1']
--------------------------------------------------

--- DEBUG_CROSS for F2 ---
DEBUG_

In [10]:
# Convert global lists to DataFrames
# locus_data_list_global and recombination_data_list_global are returned by simulate_generations
locus_level_df = pd.DataFrame(locus_data_list_global)
chromatid_recomb_df = pd.DataFrame(recombination_data_list_global)

print("\nLocus-Level Genetic Data DataFrame (First 10 rows)")
print(locus_level_df.head(10)) # Shows the first 10 rows of the DataFrame

print("\n Locus-Level Genetic Data DataFrame Info")
locus_level_df.info() # Provides a concise summary including column dtypes, non-null values, and memory usage
print(f"\nTotal rows in Locus-Level Data: {locus_level_df.shape[0]}") # Shows the total number of rows (data points)


print("\nChromatid Recombination Data DataFrame (First 10 rows)")
print(chromatid_recomb_df.head(10)) # Shows the first 10 rows of the recombination DataFrame

print("\nChromatid Recombination Data DataFrame Info")
chromatid_recomb_df.info()
print(f"\nTotal rows in Chromatid Recombination Data: {chromatid_recomb_df.shape[0]}")


# -- Save the DataFrames to CSV files for permanent storage ---
output_dir = 'simulation_results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

locus_level_output_path = os.path.join(output_dir, 'locus_level_genotypes_with_pedigree.csv')
chromatid_recomb_output_path = os.path.join(output_dir, 'chromatid_recombination_data.csv')

print("\nSaving DataFrames to CSV files...")
locus_level_df.to_csv(locus_level_output_path, index=False)
chromatid_recomb_df.to_csv(chromatid_recomb_output_path, index=False)
print(f"Locus-level data saved to: {locus_level_output_path}")
print(f"Chromatid recombination data saved to: {chromatid_recomb_output_path}")
print("-" * 50)


Locus-Level Genetic Data DataFrame (First 10 rows)
  generation  individual_id  diploid_chr_id  locus_position  \
0        P_A              1               1               0   
1        P_A              1               1               1   
2        P_A              1               1               2   
3        P_A              1               1               3   
4        P_A              1               1               4   
5        P_A              1               1               5   
6        P_A              1               1               6   
7        P_A              1               1               7   
8        P_A              1               1               8   
9        P_A              1               1               9   

   genotype_allele_A  genotype_allele_B ancestry_chromatid1_pop  \
0                  2                  2                     P_A   
1                  2                  2                     P_A   
2                  2                  2              

In [12]:
# Example absolute path on Windows
output_directory = r"C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\datafr" # Use 'r' for raw string to avoid issues with backslashes

# Create the directory if it doesn't exist, including any necessary parent directories
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True) # exist_ok=True prevents an error if the directory already exists

locus_level_df.to_csv(os.path.join(output_directory, "locus_level_genotypes_with_ancestry_pedigree_20.csv"), index=False)
chromatid_recomb_df.to_csv(os.path.join(output_directory, "chromatid_recombination_data_with_ancestry.csv"), index=False)
print(f"Locus level data saved to: {os.path.join(output_directory, 'locus_level_genotypes_with_ancestry_pedigree_20.csv')}")
print(f"Chromatid recombination data saved to: {os.path.join(output_directory, 'chromatid_recombination_data_with_ancestry.csv')}")

Locus level data saved to: C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\datafr\locus_level_genotypes_with_ancestry_pedigree_20.csv
Chromatid recombination data saved to: C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\datafr\chromatid_recombination_data_with_ancestry.csv
